---

# 스프린트미션16 4팀_김명환

## 1. 기본 라이브러리 / 함수
### 1.1. 라이브러리

In [1]:
import importlib
import sys
import subprocess

def install_if_missing(package_name, module_name=None, index_url=None):
    # module_name이 없으면 package_name을 그대로 사용
    module_name = module_name or package_name.replace("-", "_")

    if importlib.util.find_spec(module_name) is None:
        print(f"{package_name} 설치 중...")
        cmd = [sys.executable, "-m", "pip", "install"]
        if index_url:
            cmd += ["--index-url", index_url]
        cmd.append(package_name)
        subprocess.check_call(cmd)
    else:
        print(f"{package_name} 이미 설치되어 있음.")

# 사용 예시
install_if_missing("helper-plot-hangul", "helper_plot_hangul", "https://test.pypi.org/simple/")
install_if_missing("helper-utils", "helper_utils", "https://test.pypi.org/simple/")


helper-plot-hangul 이미 설치되어 있음.
helper-utils 이미 설치되어 있음.


In [2]:
# import importlib
# from helper_plot_hangul import helper_plot_hangul
# importlib.reload(helper_plot_hangul)

# import helper_utils.helper_logger as helper_logger
# importlib.reload(helper_logger)

# import helper_utils.helper_utils_colab as helper_utils_colab
# importlib.reload(helper_utils_colab)

from helper_plot_hangul import *
from helper_utils.helper_logger import *
from helper_utils.helper_utils_colab import *
from helper_utils.helper_utils_print import *
from helper_utils.helper_pandas import *


In [3]:
# 기본 라이브러리

# --- Scikit-learn: 데이터 전처리, 모델, 평가 ---
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import (
    fetch_california_housing, load_iris, make_moons, make_circles,
    load_breast_cancer, load_wine
)
from sklearn import datasets
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.metrics import average_precision_score

# --- 기타 라이브러리 ---
from PIL import Image
from PIL import ImageFilter
from PIL import ImageDraw
import albumentations as A
import IPython.display
#from tqdm import tqdm
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# --- PyTorch: 딥러닝 관련 ---
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision.datasets import CocoDetection
from torchvision.transforms import functional as TF
from torch.nn import CrossEntropyLoss
from collections import OrderedDict

# --- 기타 ---
import re
import os
import sys
import copy
import json
import math
import random
import yaml
import shutil
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from datetime import datetime
from datetime import timezone, timedelta
import pytz
__kst = pytz.timezone('Asia/Seoul')

# GPU 설정
__device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
__device_cpu = torch.device('cpu')

  # 재현 가능한 결과를 위해
np.random.seed(42)
torch.manual_seed(42)
if __device == 'cuda':
    torch.cuda.manual_seed_all(42)

print(f"라이브러리 로드 완료 사용장치:{__device}")

라이브러리 로드 완료 사용장치:cpu


### > 설정 < 플레그

In [4]:
DEBUG_ON = False if IS_COLAB else True
DEBUG_ON = False
TRAIN_ON = False
logger.info(f"IS_COLAB={IS_COLAB}")
logger.info(f"DEBUG_ON={DEBUG_ON}")


2025-12-08 15:33:28 I [helper_pandas:4] - IS_COLAB=False
2025-12-08 15:33:28 I [helper_pandas:5] - DEBUG_ON=False


## 2. 데이터 로드

- list.txt 파싱

In [5]:
root_cache_path = my_cache()
root_my_driver = my_driver()

logger.debug(f"root_cache_path: {root_cache_path}")
logger.debug(f"root_my_driver: {root_my_driver}")

### Yolo DataSet

In [6]:
import os, sys
import importlib
sys.path.insert(0, os.getcwd())

# 기존 모듈 완전 제거
if 'yolo_eval' in sys.modules:
    del sys.modules['yolo_eval']
    
# 하위 모듈도 제거
for key in list(sys.modules.keys()):
    if key.startswith('yolo_eval.'):
        del sys.modules[key]

# 새로 임포트
from yolo_eval import *

logger.info("YOLOEvaluator 클래스 로드 완료 (**kwargs 지원 추가)")

2025-12-08 15:33:29 I [helper_logger:58] - EvaluationMetrics 클래스 로드 완료
2025-12-08 15:33:29 I [helper_logger:24] - PredictionResult 클래스 로드 완료
2025-12-08 15:33:29 I [helper_logger:682] - QuantizedModelWrapper 모듈 로드 완료
2025-12-08 15:33:29 I [helper_logger:450] - YOLOEvaluator 클래스 로드 완료
2025-12-08 15:33:29 I [helper_logger:255] - YOLOEvaluationPipeline 클래스 로드 완료
2025-12-08 15:33:29 I [helper_pandas:17] - YOLOEvaluator 클래스 로드 완료 (**kwargs 지원 추가)


In [7]:
logger.setLevel(logging.INFO)
yolo_dataset_path = my_cache_path("yolo", "the-oxfordiiit-pet-dataset_eval")
#yaml_path, train_df, valid_df, test_df, validation_results = oxfordiit_pet_to_yolo(max_samples_per_split=(30, 20, 10),
yaml_path, train_df, valid_df, test_df, validation_results = oxfordiit_pet_to_yolo(max_samples_per_split=None,
                                                                                   label_mode="species",
                                                                                   output_dir=Path(yolo_dataset_path))
logger.setLevel(logging.DEBUG)

Processing train:  21%|██        | 535/2576 [00:02<00:10, 203.10it/s]2025-12-08 15:33:33,794 - yolo_eval.oxfordiiit_pet_dataset - WARNING - XML 파일 없음: C:\Users\sw1\.cache\kagglehub\datasets\devdgohil\the-oxfordiiit-pet-dataset\versions\2\annotations\annotations\xmls\Egyptian_Mau_14.xml
2025-12-08 15:33:33,871 - yolo_eval.oxfordiiit_pet_dataset - WARNING - XML 파일 없음: C:\Users\sw1\.cache\kagglehub\datasets\devdgohil\the-oxfordiiit-pet-dataset\versions\2\annotations\annotations\xmls\Egyptian_Mau_156.xml
Processing test: 100%|██████████| 368/368 [00:01<00:00, 189.48it/s]
2025-12-08 15:33:50,837 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\labels\train\Abyssinian_104.txt
2025-12-08 15:33:59,753 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\labels\train\Bengal_111.txt
2025-12-08 15:34:00,655 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\c

## 3. 평가

In [8]:
EVAL_MODEL = {}
EVAL_MODEL['YOLOv8m_baseline'] = True
EVAL_MODEL['YOLOv8m_int8_openvino'] = True
EVAL_MODEL['YOLOv8m_ONNX_FP32'] = True
EVAL_MODEL['YOLOv8m_ONNX_FP16'] = True
EVAL_MODEL['YOLOv8m_ONNX_int8_qdq'] = True


In [9]:
logger.info(f"yaml_path: {yaml_path}")
pipeline = YOLOEvaluationPipeline(
    yaml_path=yaml_path,
    device=str(__device)
)


2025-12-08 15:34:54 I [helper_pandas:1] - yaml_path: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\data.yaml


In [10]:
logger.setLevel(logging.INFO)
yolov8m_path = my_driver_path('modeling', 'model', 'modeling16', 'yolov8m_20251205_2005')
logger.setLevel(logging.DEBUG)


2025-12-08 15:34:54 I [helper_utils_colab:384] - Created directory: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005


In [11]:
# 기본
if EVAL_MODEL['YOLOv8m_baseline']:
    yolov8m_best_path = my_driver_path(yolov8m_path, 'weights', 'best.pt', create=False)
    logger.info(f"yolov8m_best_path: {yolov8m_best_path}")
    pipeline.add_model(
        model_path=yolov8m_best_path,
        model_name="YOLOv8m_baseline",
        verbose=False,
        model_type='yolo_pt',
    )

2025-12-08 15:34:54 I [helper_pandas:4] - yolov8m_best_path: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best.pt
2025-12-08 15:34:54 I [helper_logger:95] - 모델 추가: YOLOv8m_baseline (타입: yolo_pt)
2025-12-08 15:34:54 I [helper_logger:75] - 모델 로딩 중 (타입: yolo_pt): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best.pt
2025-12-08 15:34:54 I [helper_logger:184] - 모델 로드 완료: YOLOv8m_baseline
2025-12-08 15:34:54 I [helper_logger:195] - 데이터셋 경로: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval
2025-12-08 15:34:54 I [helper_logger:196] - 테스트 이미지: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\images/test
2025-12-08 15:34:54 I [helper_logger:197] - 테스트 라벨: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\labels\test


In [12]:
# Yolo 8 int 8양자화
if EVAL_MODEL['YOLOv8m_int8_openvino']:
    output_openvino_int8_path = my_driver_path(yolov8m_path, 'weights', 'best_int8_openvino_model', create=False)
    logger.info(f"output_openvino_int8_path: {output_openvino_int8_path}")

    # OpenVINO 모델은 디렉토리 경로를 전달해야 합니다
    # Ultralytics는 '_openvino_model' suffix로 OpenVINO 형식을 감지합니다
    pipeline.add_model(
        model_path=output_openvino_int8_path,  # 디렉토리 경로 (수정됨)
        model_name="YOLOv8m_int8_openvino",
        verbose=False,
        model_type='openvino',
    )


2025-12-08 15:34:54 I [helper_pandas:4] - output_openvino_int8_path: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best_int8_openvino_model
2025-12-08 15:34:54 I [helper_logger:95] - 모델 추가: YOLOv8m_int8_openvino (타입: openvino)
2025-12-08 15:34:54 I [helper_logger:75] - 모델 로딩 중 (타입: openvino): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best_int8_openvino_model
WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
2025-12-08 15:34:54 I [helper_logger:179] - OpenVINO 모델 로드 완료
2025-12-08 15:34:54 I [helper_logger:184] - 모델 로드 완료: YOLOv8m_int8_openvino
2025-12-08 15:34:54 I [helper_logger:195] - 데이터셋 경로: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval
2025-12-08 15:34:54 I [helper_logger:196] - 테스트 이미지: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\images/test
2025-12-08 15:34:54 I [helper_logger:197] -

In [13]:
# YOLOv8m_ONNX_FP32
if EVAL_MODEL['YOLOv8m_ONNX_FP32']:
    yolov8m_onnx_fp32_path = my_driver_path(yolov8m_path, 'weights', 'out_onnx_fp32', 'yolov8m_fp32.onnx', create=False)
    pipeline.add_model(
        model_path=yolov8m_onnx_fp32_path,
        model_name="YOLOv8m_ONNX_FP32",
        verbose=False,
        model_type='onnx',
    )

2025-12-08 15:34:54 I [helper_logger:95] - 모델 추가: YOLOv8m_ONNX_FP32 (타입: onnx)
2025-12-08 15:34:54 I [helper_logger:75] - 모델 로딩 중 (타입: onnx): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_fp32\yolov8m_fp32.onnx
2025-12-08 15:34:54 I [helper_logger:83] - ONNX 모델을 Ultralytics YOLO로 로딩
2025-12-08 15:34:54 I [helper_logger:86] - ONNX 모델 크기: 98.72 MB
2025-12-08 15:34:54 I [helper_logger:184] - 모델 로드 완료: YOLOv8m_ONNX_FP32
2025-12-08 15:34:54 I [helper_logger:195] - 데이터셋 경로: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval
2025-12-08 15:34:54 I [helper_logger:196] - 테스트 이미지: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\images/test
2025-12-08 15:34:54 I [helper_logger:197] - 테스트 라벨: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\labels\test


In [14]:
# YOLOv8m_ONNX_FP16
if EVAL_MODEL['YOLOv8m_ONNX_FP16']:
    yolov8m_onnx_fp16_path = my_driver_path(yolov8m_path, 'weights', 'out_onnx_fp16', 'yolov8m_fp16.onnx', create=False)
    pipeline.add_model(
        model_path=yolov8m_onnx_fp16_path,
        model_name="YOLOv8m_ONNX_FP16",
        verbose=False,
        model_type='onnx',
    )


2025-12-08 15:34:54 I [helper_logger:95] - 모델 추가: YOLOv8m_ONNX_FP16 (타입: onnx)
2025-12-08 15:34:54 I [helper_logger:75] - 모델 로딩 중 (타입: onnx): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_fp16\yolov8m_fp16.onnx
2025-12-08 15:34:54 I [helper_logger:83] - ONNX 모델을 Ultralytics YOLO로 로딩
2025-12-08 15:34:54 I [helper_logger:86] - ONNX 모델 크기: 98.72 MB
2025-12-08 15:34:54 I [helper_logger:184] - 모델 로드 완료: YOLOv8m_ONNX_FP16
2025-12-08 15:34:54 I [helper_logger:195] - 데이터셋 경로: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval
2025-12-08 15:34:54 I [helper_logger:196] - 테스트 이미지: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\images/test
2025-12-08 15:34:54 I [helper_logger:197] - 테스트 라벨: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\labels\test


In [15]:
# YOLOv8m_ONNX_int8
if EVAL_MODEL['YOLOv8m_ONNX_int8_qdq']:
    yolov8m_onnx_int8_path = my_driver_path(yolov8m_path, 'weights', 'out_onnx_int8_qdq', 'yolov8m_int8_qdq.onnx', create=False)
    pipeline.add_model(
        model_path=yolov8m_onnx_int8_path,
        model_name="YOLOv8m_ONNX_int8_qdq",
        verbose=False,
        model_type='onnx',
    )


2025-12-08 15:34:54 I [helper_logger:95] - 모델 추가: YOLOv8m_ONNX_int8_qdq (타입: onnx)
2025-12-08 15:34:54 I [helper_logger:75] - 모델 로딩 중 (타입: onnx): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_int8_qdq\yolov8m_int8_qdq.onnx
2025-12-08 15:34:54 I [helper_logger:83] - ONNX 모델을 Ultralytics YOLO로 로딩
2025-12-08 15:34:54 I [helper_logger:86] - ONNX 모델 크기: 25.30 MB
2025-12-08 15:34:54 I [helper_logger:184] - 모델 로드 완료: YOLOv8m_ONNX_int8_qdq
2025-12-08 15:34:54 I [helper_logger:195] - 데이터셋 경로: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval
2025-12-08 15:34:54 I [helper_logger:196] - 테스트 이미지: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\images/test
2025-12-08 15:34:55 I [helper_logger:197] - 테스트 라벨: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\labels\test


In [16]:
# yolov8m_best_path = my_driver_path(yolov8m_path, 'weights', 'best.pt', create=False)
# evaluator = YOLOEvaluator(
#     model_path=yolov8m_best_path,
#     yaml_path=yaml_path,
#     model_name="yolov8m_fp32",
#     device=str(__device),
#     verbose=False,
#     model_type="yolo_pt",
# )
# evaluator.model


In [17]:

# yolov8m_onnx_fp32_path = my_driver_path(yolov8m_path, 'weights', 'out_onnx_fp32', 'yolov8m_fp32.onnx', create=False)
# evaluator = YOLOEvaluator(
#     model_path=yolov8m_onnx_fp32_path,
#     yaml_path=yaml_path,
#     model_name="yolov8m_fp32",
#     device=str(__device),
#     verbose=False,
#     model_type="onnx",
# )
# evaluator.model


In [18]:
# 전체 평가 실행
yolov8m_result_path = my_driver_path(yolov8m_path, 'weights', 'result', create=True)
results = pipeline.run_full_evaluation(
    val_kwargs={
        'project': yolov8m_result_path,
        'split': 'test',
        'imgsz': 640,
        'batch': 1,      # ONNX 고정 shape 문제로 배치 크기 1로 축소
        'conf': 0.001,
        'iou': 0.45,
        'rect': False    # rectangular inference 비활성화 (동일 크기 강제)
    },
)
# results = pipeline.run_full_evaluation(
#     val_kwargs={
#         'project': yolov8m_result_path,  # custom_results/ 디렉터리에
#         # 'conf': mission_16_yolo_yaml,
#         # 'name': 'baseline_test',      # baseline_test/ 하위 폴더로 저장
#         'split': 'test',
#         'imgsz': 640,
#         'batch': 16,
#         'conf': 0.25,
#         'iou': 0.75
#     },
#     # pred_max_images=100,
#     pred_conf=0.25
# )


2025-12-08 15:34:55 I [helper_utils_colab:384] - Created directory: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\result
모델 검증 시작

[YOLOv8m_baseline] 검증 중...
2025-12-08 15:34:55 I [helper_logger:250] - 모델 검증 시작: split=test, imgsz=640, batch=1
2025-12-08 15:34:55 I [helper_logger:251] - 모델 검증 시작: self.model_type=yolo_pt
2025-12-08 15:34:55 I [helper_logger:252] - 모델 검증 시작: self.model_config=None
Ultralytics 8.3.235  Python-3.10.19 torch-2.5.1+cpu CPU (12th Gen Intel Core(TM) i7-1260P)
Model summary (fused): 92 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 75.441.5 MB/s, size: 147.7 KB)
val: Scanning D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\labels\test... 368 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 368/368 431.7it/s 0.9s0.0s
val: New cache created: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset_eval\labels\test.cache
                 Class     Images  Instances      Box(P  

이미지 예측 중:   0%|          | 0/368 [00:00<?, ?it/s]

2025-12-08 16:15:31 I [helper_logger:357] - 예측 완료: 368개 이미지
[YOLOv8m_int8_openvino] 예측 중...
2025-12-08 16:15:31 I [helper_logger:322] - 이미지 예측 시작: max_images=None


이미지 예측 중:   0%|          | 0/368 [00:00<?, ?it/s]

Loading D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best_int8_openvino_model for OpenVINO inference...
Using OpenVINO LATENCY mode for batch=1 inference on (CPU)...
2025-12-08 16:17:11 I [helper_logger:357] - 예측 완료: 368개 이미지
[YOLOv8m_ONNX_FP32] 예측 중...
2025-12-08 16:17:11 I [helper_logger:322] - 이미지 예측 시작: max_images=None


...

Loading D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_fp32\yolov8m_fp32.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.23.2 CPUExecutionProvider
2025-12-08 16:22:55 I [helper_logger:357] - 예측 완료: 368개 이미지
[YOLOv8m_ONNX_FP16] 예측 중...
2025-12-08 16:22:55 I [helper_logger:322] - 이미지 예측 시작: max_images=None


이미지 예측 중:   0%|          | 0/368 [00:00<?, ?it/s]

Loading D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_fp16\yolov8m_fp16.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.23.2 CPUExecutionProvider
2025-12-08 16:28:03 I [helper_logger:357] - 예측 완료: 368개 이미지
[YOLOv8m_ONNX_int8_qdq] 예측 중...
2025-12-08 16:28:03 I [helper_logger:322] - 이미지 예측 시작: max_images=None


이미지 예측 중:   0%|          | 0/368 [00:00<?, ?it/s]

Loading D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_int8_qdq\yolov8m_int8_qdq.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.23.2 CPUExecutionProvider
2025-12-08 16:36:40 I [helper_logger:357] - 예측 완료: 368개 이미지


In [19]:
# 결과 출력
pipeline.print_summary()

평가 결과: YOLOv8m_baseline
mAP50: 0.9942
mAP50-95: 0.9178
정밀도(Precision): 0.9942
재현율(Recall): 0.9783
추론 시간(평균): 1101.03ms
--------------------------------------------------------------------------------
테스트된 총 이미지 수: 368
GT 박스가 있는 이미지 수: 368
예측이 있는 이미지 수: 368
이미지당 평균 GT 박스 수: 1.00
이미지당 평균 예측 수: 1.03
평가 결과: YOLOv8m_int8_openvino
mAP50: 0.9942
mAP50-95: 0.9105
정밀도(Precision): 0.9917
재현율(Recall): 0.9776
추론 시간(평균): 266.57ms
--------------------------------------------------------------------------------
테스트된 총 이미지 수: 368
GT 박스가 있는 이미지 수: 368
예측이 있는 이미지 수: 368
이미지당 평균 GT 박스 수: 1.00
이미지당 평균 예측 수: 1.03
평가 결과: YOLOv8m_ONNX_FP32
mAP50: 0.9942
mAP50-95: 0.9178
정밀도(Precision): 0.9942
재현율(Recall): 0.9783
추론 시간(평균): 926.84ms
--------------------------------------------------------------------------------
테스트된 총 이미지 수: 368
GT 박스가 있는 이미지 수: 368
예측이 있는 이미지 수: 368
이미지당 평균 GT 박스 수: 1.00
이미지당 평균 예측 수: 1.03
평가 결과: YOLOv8m_ONNX_FP16
mAP50: 0.9942
mAP50-95: 0.9178
정밀도(Precision): 0.9942
재현율(Recall): 0.9783
추론 

In [29]:
# 비교 DataFrame 생성
comparison_df = pipeline.get_comparison_dataframe()
logger.info("모델 비교 결과:")
display(comparison_df)

2025-12-08 16:40:12 I [helper_pandas:3] - 모델 비교 결과:


,model_name,mAP50,mAP50-95,precision,recall,inference_time_ms,total_images,images_with_gt,images_with_pred,avg_gt_boxes,avg_pred_boxes
0,YOLOv8m_baseline,0.994249,0.917792,0.994234,0.978261,1101.029359,368,368,368,1.0,1.032609
2,YOLOv8m_ONNX_FP32,0.994249,0.917792,0.994234,0.978261,926.842964,368,368,368,1.0,1.032609
3,YOLOv8m_ONNX_FP16,0.994249,0.917792,0.994234,0.978261,831.753450,368,368,368,1.0,1.032609
1,YOLOv8m_int8_openvino,0.994220,0.910524,0.991730,0.977610,266.569612,368,368,368,1.0,1.032609
4,YOLOv8m_ONNX_int8_qdq,0.993784,0.866676,0.988138,0.978261,1399.696517,368,368,366,1.0,1.040761
